# Unified User List Construction

Match emails across GoVocal users and Typeform survey responses to build:
1. **Unified Users** — deduplicated list of all identified users across platforms
2. **Anonymous Users** — contributions (ideas, comments, survey responses) with no identifiable author

In [7]:
import json
import uuid
from collections import Counter
from pathlib import Path

DATA_DIR = Path("../data")

TYPEFORM_NAME_FIELD = "7c41d300-10a5-4393-a277-3c9bb936098a"
TYPEFORM_EMAIL_FIELD = "6e606303-4fb5-47be-a692-767dc43a447c"
FORM_IDS = ["KdHzkJeL", "PmPIQkd8", "YcnYy8ah"]

def load_json(filename):
    with open(DATA_DIR / filename) as f:
        return json.load(f)

def normalize_email(email):
    if not email or not email.strip():
        return None
    return email.strip().lower()

def make_user(email, **kwargs):
    return {
        "unified_id": str(uuid.uuid4()),
        "email": email,
        "first_name": None, "last_name": None, "name": None,
        "sources": [],
        "govocal_user_id": None, "govocal_created_at": None,
        "typeform_response_ids": [],
        "govocal_ideas_count": 0, "govocal_comments_count": 0, "govocal_reactions_count": 0,
        **kwargs,
    }

In [8]:
# Load all data files
gv_users = load_json("govocal_users.json")
gv_ideas = load_json("govocal_ideas.json")
gv_comments = load_json("govocal_comments.json")
gv_reactions = load_json("govocal_reactions.json")
tf_responses = {fid: load_json(f"typeform_{fid}.json") for fid in FORM_IDS}

for name, data in [("users", gv_users), ("ideas", gv_ideas), ("comments", gv_comments), ("reactions", gv_reactions)]:
    print(f"GoVocal {name}: {len(data)}")
for fid, resps in tf_responses.items():
    print(f"Typeform {fid}: {len(resps)} responses")

GoVocal users: 2754
GoVocal ideas: 3243
GoVocal comments: 310
GoVocal reactions: 22064
Typeform KdHzkJeL: 4960 responses
Typeform PmPIQkd8: 2604 responses
Typeform YcnYy8ah: 2910 responses


In [9]:
# Build unified user list — seed from GoVocal, then merge Typeform by email

unified_by_email = {}
govocal_id_to_email = {}

for user in gv_users:
    email = normalize_email(user.get("email"))
    if not email:
        continue
    govocal_id_to_email[user["id"]] = email
    unified_by_email[email] = make_user(
        email,
        first_name=user.get("first_name"), last_name=user.get("last_name"),
        sources=["govocal"], govocal_user_id=user["id"],
        govocal_created_at=user.get("created_at"),
    )

for form_id, responses in tf_responses.items():
    for resp in responses:
        email = normalize_email(resp.get("email")) or normalize_email(resp.get(TYPEFORM_EMAIL_FIELD))
        if not email:
            continue
        tf_ref = {"form_id": form_id, "response_id": resp["response_id"]}
        source_tag = f"typeform_{form_id}"

        if email in unified_by_email:
            user = unified_by_email[email]
            if source_tag not in user["sources"]:
                user["sources"].append(source_tag)
            user["typeform_response_ids"].append(tf_ref)
            if not user["first_name"] and not user["name"]:
                user["name"] = resp.get(TYPEFORM_NAME_FIELD)
        else:
            unified_by_email[email] = make_user(
                email, name=resp.get(TYPEFORM_NAME_FIELD),
                sources=[source_tag], typeform_response_ids=[tf_ref],
            )

print(f"Unified users: {len(unified_by_email)}")

Unified users: 5205


In [10]:
# Enrich with GoVocal activity counts

for data, id_field, count_field in [
    (gv_ideas, "author_id", "govocal_ideas_count"),
    (gv_comments, "author_id", "govocal_comments_count"),
    (gv_reactions, "user_id", "govocal_reactions_count"),
]:
    for record in data:
        uid = record.get(id_field)
        if uid in govocal_id_to_email:
            unified_by_email[govocal_id_to_email[uid]][count_field] += 1

active = sum(1 for u in unified_by_email.values()
             if u["govocal_ideas_count"] + u["govocal_comments_count"] + u["govocal_reactions_count"] > 0)
print(f"Users with GoVocal activity: {active}")

Users with GoVocal activity: 1812


In [11]:
# Build anonymous contributions list

anonymous = []

for data, source, ctype, title_key in [
    (gv_ideas, "govocal_ideas", "idea", "title"),
    (gv_comments, "govocal_comments", "comment", None),
]:
    for rec in data:
        if rec.get("author_id") is None:
            anonymous.append({
                "source": source, "record_id": rec["id"], "content_type": ctype,
                "title": rec.get(title_key) if title_key else None,
                "timestamp": rec.get("created_at"),
            })

for form_id, responses in tf_responses.items():
    for resp in responses:
        email = normalize_email(resp.get("email")) or normalize_email(resp.get(TYPEFORM_EMAIL_FIELD))
        if not email:
            anonymous.append({
                "source": f"typeform_{form_id}", "record_id": resp["response_id"],
                "content_type": "survey_response", "title": None,
                "timestamp": resp.get("landed_at"),
                "response_type": resp.get("response_type"),
                "name": resp.get(TYPEFORM_NAME_FIELD),
            })

source_counts = Counter(a["source"] for a in anonymous)
print(f"Anonymous contributions: {len(anonymous)}")
for source, count in sorted(source_counts.items()):
    print(f"  {source}: {count}")

Anonymous contributions: 9087
  govocal_comments: 54
  govocal_ideas: 2440
  typeform_KdHzkJeL: 3053
  typeform_PmPIQkd8: 1971
  typeform_YcnYy8ah: 1569


In [12]:
# Save and summarize

unified_users = list(unified_by_email.values())

for filename, data in [("unified_users.json", unified_users), ("anonymous_users.json", anonymous)]:
    with open(DATA_DIR / filename, "w") as f:
        json.dump(data, f, indent=2)

govocal_only = sum(1 for u in unified_users if u["sources"] == ["govocal"])
typeform_only = sum(1 for u in unified_users if "govocal" not in u["sources"])
cross_platform = sum(1 for u in unified_users if "govocal" in u["sources"] and len(u["sources"]) > 1)

print(f"Total unified users:        {len(unified_users)}")
print(f"  GoVocal only:             {govocal_only}")
print(f"  Typeform only:            {typeform_only}")
print(f"  Cross-platform (matched): {cross_platform}")
print(f"Anonymous contributions:     {len(anonymous)}")
for source, count in sorted(source_counts.items()):
    print(f"  {source}: {count}")

Total unified users:        5205
  GoVocal only:             1752
  Typeform only:            2451
  Cross-platform (matched): 1002
Anonymous contributions:     9087
  govocal_comments: 54
  govocal_ideas: 2440
  typeform_KdHzkJeL: 3053
  typeform_PmPIQkd8: 1971
  typeform_YcnYy8ah: 1569
